# Basic Image Classification with PyTorch

This tutorial guides you through creating a basic neural network to classify images using PyTorch. We will use the **FashionMNIST** dataset, which consists of 28x28 grayscale images of 10 fashion categories (t-shirts, trousers, pullovers, dresses, coats, sandals, shirts, sneakers, bags, and ankle boots).

We will cover:
1.  **Setup**: Importing libraries and checking for GPU.
2.  **Data**: Loading and transforming the dataset.
3.  **Model**: Building a simple Neural Network.
4.  **Training**: The standard training loop.
5.  **Evaluation**: Testing the model's accuracy.

## 1. Setup

First, we import `torch` and `torchvision`. We also check if a GPU (or MPS on Mac) is available to speed up training.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

## 2. Data Preparation

We use `datasets.FashionMNIST` to download the data. 
- **root**: Where to store the data.
- **train**: True for training data, False for test data.
- **download**: True ensures it downloads if not present.
- **transform**: `ToTensor()` converts images (PIL or numpy) into PyTorch Tensors and scales features to [0.0, 1.0].

We then wrap these datasets in a `DataLoader`. The DataLoader handles batching (grouping samples together) and shuffling (randomizing order) which are crucial for stable training.

In [ ]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

## 3. Define the Model

We define our neural network by subclassing `nn.Module`.

1.  **`__init__`**: We define the layers.
    - `Flatten`: Converts the 2D 28x28 image into a 1D vector of 784 pixels.
    - `Linear`: A fully connected layer that transforms the input. `Linear(784, 512)` means it takes 784 inputs and outputs 512 features.
    - `ReLU`: The activation function. It introduces non-linearity, allowing the network to learn complex patterns. It essentially computes `max(0, x)`.
    - The final layer outputs 10 values, one for each class.

2.  **`forward`**: This function defines how data passes through the network. We apply the flattening, then the stack of linear and relu layers.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

## 4. Optimizing the Model Parameters

To train a model, we need a **Loss Function** and an **Optimizer**.
- **Loss Function**: Measures how "wrong" the model's predictions are. We use `CrossEntropyLoss` for classification tasks.
- **Optimizer**: Updates the model's parameters (weights) based on the gradients computed from the loss. We use `SGD` (Stochastic Gradient Descent).

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

## 5. Training Loop

In a single training loop, the model makes predictions on the training dataset (fed to it in batches), and backpropagates the prediction error to adjust the model's parameters.

**The Steps:**
1.  **Compute prediction (`pred`) and loss**: Pass data through model, compare to True label.
2.  **Backpropagation**:
    - `optimizer.zero_grad()`: Reset gradients from previous step. Crucial, as PyTorch accumulates gradients by default.
    - `loss.backward()`: Compute gradients of the loss with respect to model parameters.
    - `optimizer.step()`: Adjust parameters using the gradients.

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train() # Set model to training mode
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

## 6. Evaluation Loop

We also check the model's performance against the test dataset to ensure it is learning.
Note `torch.no_grad()`: We don't need to compute gradients during evaluation, so we disable it to save memory and computation.

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval() # Set model to evaluation mode
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## 7. Execution

We run the training and testing for a number of **epochs** (passes over the dataset).

In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")